In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import numpy as np
import time
import psutil
import os

print(f"TensorFlow версия: {tf.__version__}")
print(f"Доступные устройства: {tf.config.list_physical_devices()}")
print(f"Процессор: {psutil.cpu_freq().max:.0f} MHz, Ядер: {psutil.cpu_count(logical=True)}")
print(f"ОЗУ: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print("="*60)

# Оптимально для Ryzen AI 5 330 (6 физических ядер, 12 логических)
os.environ["OMP_NUM_THREADS"] = "8"          # 8 логических ядер (оптимально)
os.environ["TF_NUM_INTRAOP_THREADS"] = "6"   # Физические ядра
os.environ["TF_NUM_INTEROP_THREADS"] = "2"   # 2 для параллельных операций
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"    # Включить Intel oneDNN

os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=2"

# ============================================================
# 1. ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ (чтобы не тратить время на загрузку)
# ============================================================
print("\n[1] Генерация синтетических данных...")
X_train = np.random.rand(10000, 28, 28, 1).astype(np.float32)  # 10k изображений 28x28
y_train = np.random.randint(0, 10, size=(10000,))
X_test = np.random.rand(2000, 28, 28, 1).astype(np.float32)
y_test = np.random.randint(0, 10, size=(2000,))

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"  Данные готовы: X_train {X_train.shape}, y_train {y_train_cat.shape}")

# ============================================================
# 2. МОДЕЛЬ #1: ВАША БАЗОВАЯ СЕТЬ (для сравнения)
# ============================================================
print("\n[2] Тестирование базовой модели...")
model_simple = Sequential()
model_simple.add(Dense(196, input_shape=(784,)))
model_simple.add(Activation('relu'))
model_simple.add(Dropout(0.5))
model_simple.add(Dense(10))
model_simple.add(Activation('softmax'))

model_simple.compile(optimizer=Adam(learning_rate=0.001),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Ресемплируем данные для этой модели (flatten)
X_train_flat = X_train.reshape(10000, 784)
X_test_flat = X_test.reshape(2000, 784)

start = time.time()
history_simple = model_simple.fit(X_train_flat, y_train_cat,
                                  epochs=5,
                                  batch_size=128,
                                  validation_data=(X_test_flat, y_test_cat),
                                  verbose=1)
time_simple = time.time() - start
print(f"  Базовая модель: {time_simple:.2f} сек за 5 эпох")

# ============================================================
# 3. МОДЕЛЬ #2: СВЁРТОЧНАЯ СЕТЬ (CNN) — намного сложнее
# ============================================================
print("\n[3] Тестирование свёрточной сети (CNN)...")
model_cnn = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model_cnn.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

model_cnn.summary()

start = time.time()
history_cnn = model_cnn.fit(X_train, y_train_cat,
                            epochs=5,
                            batch_size=64,
                            validation_data=(X_test, y_test_cat),
                            verbose=1)
time_cnn = time.time() - start
print(f"  CNN модель: {time_cnn:.2f} сек за 5 эпох")

# ============================================================
# 4. МОДЕЛЬ #3: КРУПНАЯ ПОЛНОСВЯЗНАЯ СЕТЬ (самая тяжёлая)
# ============================================================
print("\n[4] Тестирование крупной полносвязной сети...")
model_large = Sequential([
    Dense(1024, input_shape=(784,), activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    Dropout(0.3),
    
    Dense(10, activation='softmax')
])

model_large.compile(optimizer=Adam(learning_rate=0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

model_large.summary()

start = time.time()
history_large = model_large.fit(X_train_flat, y_train_cat,
                                epochs=5,
                                batch_size=64,
                                validation_data=(X_test_flat, y_test_cat),
                                verbose=1)
time_large = time.time() - start
print(f"  Крупная полносвязная модель: {time_large:.2f} сек за 5 эпох")

# ============================================================
# 5. СРАВНИТЕЛЬНЫЙ АНАЛИЗ
# ============================================================
print("\n" + "="*60)
print("📊 РЕЗУЛЬТАТЫ БЕНЧМАРКА")
print("="*60)
print(f"Базовая модель (Dense 196+10):        {time_simple:.2f} сек")
print(f"Свёрточная сеть (CNN, 5 слоёв):       {time_cnn:.2f} сек")
print(f"Крупная полносвязная (1024+512+256+128): {time_large:.2f} сек")
print("="*60)

# Оценка загруженности CPU
print("\nЗагруженность CPU в процессе работы:")
print(f"  Средняя загрузка за всё время: {psutil.cpu_percent(interval=1):.0f}%")

# Проверка на использование GPU (если вдруг)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"  ✅ Найдены GPU: {gpus}")
else:
    print("  ❌ GPU не обнаружен (работаем на CPU)")

print("\n💡 Интерпретация:")
print("  - Если CNN работает за 5-10 секунд — отличная производительность для ноутбука.")
print("  - Если CNN работает за 20-30 секунд — нормально для вашего процессора.")
print("  - Если больше минуты — стоит уменьшить batch_size или использовать меньшие модели.")

TensorFlow версия: 2.21.0
Доступные устройства: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Процессор: 2000 MHz, Ядер: 8
ОЗУ: 15.1 GB

[1] Генерация синтетических данных...
  Данные готовы: X_train (10000, 28, 28, 1), y_train (10000, 10)

[2] Тестирование базовой модели...
Epoch 1/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.1000 - loss: 2.3416 - val_accuracy: 0.1015 - val_loss: 2.3027
Epoch 2/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1012 - loss: 2.3025 - val_accuracy: 0.1000 - val_loss: 2.3024
Epoch 3/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1006 - loss: 2.3024 - val_accuracy: 0.1020 - val_loss: 2.3022
Epoch 4/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1018 - loss: 2.3023 - val_accuracy: 0.1030 - val_loss: 2.3022
Epoch 5/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1015 - loss: 2.3022 - val_accuracy: 0.1025 - val_loss: 2.3023
  Базовая модель: 2.59 сек за 5 эпох

[3] Тестирование свёрточной сети (CNN)..

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_24               │ (None, 26, 26, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 24, 24, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_25               │ (None, 24, 24, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 12, 12, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_25 (Dropout)                 │ (None, 12, 12, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 10, 10, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_26               │ (None, 10, 10, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 8, 8, 64)            │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_27               │ (None, 8, 8, 64)            │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_26 (Dropout)                 │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_29 (Dense)                     │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_28               │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_27 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_30 (Dense)                     │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 331,754 (1.27 MB)

 Trainable params: 330,858 (1.26 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_31 (Dense)                     │ (None, 1024)                │         803,840 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_29               │ (None, 1024)                │           4,096 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_28 (Dropout)                 │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_32 (Dense)                     │ (None, 512)                 │         524,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_30               │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_29 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_33 (Dense)                     │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_31               │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_30 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_34 (Dense)                     │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_31 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_35 (Dense)                     │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,501,322 (5.73 MB)

 Trainable params: 1,497,738 (5.71 MB)

 Non-trainable params: 3,584 (14.00 KB)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1005 - loss: 2.7859 - val_accuracy: 0.1115 - val_loss: 2.3497
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1281 - loss: 2.4257 - val_accuracy: 0.0970 - val_loss: 2.3485
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1550 - loss: 2.2983 - val_accuracy: 0.1010 - val_loss: 2.3557
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.1879 - loss: 2.2229 - val_accuracy: 0.0985 - val_loss: 2.3707
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.2282 - loss: 2.1454 - val_accuracy: 0.1085 - val_loss: 2.4072
  Крупная полносвязная модель: 9.31 сек за 5 эпох

📊 РЕЗУЛЬТАТЫ БЕНЧМАРКА
Базовая модель (Dense 196+10):        2.59 сек
Свёрточная сеть (CNN, 5 слоёв):       24.53 сек
Крупная полносвязная (1024+512+256+128): 9.31 сек

Загруженность CPU в процессе работы:
  Средняя загрузка за всё время: 10%
  ❌ GPU не обнаружен (работаем на CPU)

💡 Интерпретация:
  - Если CN

In [5]:
import time
import pandas as pd

def run_experiment(model, X_train, y_train, X_test, y_test, 
                   epochs=50, batch_size=64, model_name="CNN"):
    """Запуск эксперимента с замером времени и логированием"""
    
    start_time = time.time()
    
    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_data=(X_test, y_test),
                        verbose=1)
    
    elapsed = time.time() - start_time
    
    # Сохраняем результаты
    results = {
        'model': model_name,
        'epochs': epochs,
        'batch_size': batch_size,
        'train_time': elapsed,
        'final_train_acc': history.history['accuracy'][-1],
        'final_val_acc': history.history['val_accuracy'][-1],
        'final_train_loss': history.history['loss'][-1],
        'final_val_loss': history.history['val_loss'][-1]
    }
    
    return results, history

# Использование
results, history = run_experiment(model_cnn, X_train, y_train_cat, 
                                  X_test, y_test_cat, epochs=50)
print(pd.DataFrame([results]))

Epoch 1/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.1444 - loss: 2.2859 - val_accuracy: 0.0940 - val_loss: 2.3316
Epoch 2/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.1473 - loss: 2.2745 - val_accuracy: 0.1055 - val_loss: 2.3340
Epoch 3/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.1577 - loss: 2.2650 - val_accuracy: 0.1010 - val_loss: 2.3338
Epoch 4/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.1675 - loss: 2.2468 - val_accuracy: 0.1045 - val_loss: 2.3446
Epoch 5/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.1722 - loss: 2.2409 - val_accuracy: 0.0945 - val_loss: 2.3410
Epoch 6/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.1789 - loss: 2.2183 - val_accuracy: 0.0955 - val_loss: 2.3520
Epoch 7/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.2004 - loss: 2.1974 - val_accuracy: 0.0895 - val_loss: 2.3623
Epoch 8/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.2103 - loss: 2.1823 - val_accu